# Atlas - an agent for the GAIA benchmark

This is the final project I built for the Hugging Face AI Agents Course.

The goal is to build an agent that answers questions from the GAIA benchmark. These questions are easy for a person, but hard for a language model on its own, because they need web searches, reading files and a few steps of reasoning.

To get the course certificate, the agent had to answer at least 30% of 20 questions correctly.

What I used:
- smolagents (Hugging Face's agent framework)
- OpenAI's GPT-4o as the model
- Whisper to transcribe audio

## How the agent works

The agent follows a simple loop:

1. **Think** about what it needs to do
2. **Act**: write Python code that calls a tool (for example, a web search)
3. **Observe** the result
4. Repeat until it finds the answer

I used the `CodeAgent` from smolagents. The difference is that the model writes Python code instead of calling one tool at a time. This helps a lot when the question involves math or spreadsheets.

The tools the agent has:
- web search (DuckDuckGo)
- open web pages
- Wikipedia search
- audio transcription
- image analysis

## Installation

In [ ]:
%pip install -q "smolagents>=1.20" openai ddgs wikipedia-api markdownify pandas openpyxl huggingface_hub requests

## API keys

I need two keys:
- `OPENAI_API_KEY`: to use GPT-4o and Whisper
- `HF_TOKEN` (optional): to download the question files directly from the GAIA dataset on Hugging Face

The keys are typed in when the notebook runs, so they are never saved in the file.

In [ ]:
import os
from getpass import getpass

for key in ["OPENAI_API_KEY", "HF_TOKEN"]:
    if not os.getenv(key):
        value = getpass(f"{key} (press Enter to skip): ")
        if value:
            os.environ[key] = value

print("OPENAI_API_KEY:", "ok" if os.getenv("OPENAI_API_KEY") else "MISSING")
print("HF_TOKEN:", "ok" if os.getenv("HF_TOKEN") else "not set (optional)")

## Imports

In [ ]:
import os
import re
import base64
import tempfile

import requests
import pandas as pd
from markdownify import markdownify
from openai import OpenAI
from huggingface_hub import hf_hub_download
from smolagents import (
    CodeAgent,
    OpenAIServerModel,
    DuckDuckGoSearchTool,
    WikipediaSearchTool,
    tool,
)

API_URL = "https://agents-course-unit4-scoring.hf.space"  # course scoring API
MODEL_ID = "gpt-4o"
MAX_STEPS = 12

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Tools

In smolagents, to create a tool you just write a function with `@tool` on top and a docstring explaining what it does. The model reads this docstring to know when to use the tool.

I created three tools of my own:
- `visit_webpage`: opens a page and returns its text. I wrote my own because Wikipedia was blocking the default tool.
- `transcribe_audio`: transcribes audio files using Whisper.
- `analyze_image`: sends the image to GPT-4o and asks a question about it.

In [ ]:
@tool
def visit_webpage(url: str) -> str:
    """Opens a webpage and returns its content as markdown text.

    Args:
        url: The full URL of the page to open.
    """
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Atlas-GAIA-agent/1.0)"}
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    return markdownify(r.text)[:40000]


@tool
def transcribe_audio(file_path: str) -> str:
    """Transcribes an audio file (mp3, wav, m4a) to text.

    Args:
        file_path: Local path to the audio file.
    """
    with open(file_path, "rb") as f:
        result = client.audio.transcriptions.create(model="whisper-1", file=f)
    return result.text


@tool
def analyze_image(file_path: str, question: str) -> str:
    """Looks at an image file and answers a question about it.

    Args:
        file_path: Local path to the image (png, jpg).
        question: What you want to know about the image.
    """
    ext = file_path.split(".")[-1].lower()
    mime = "jpeg" if ext in ("jpg", "jpeg") else ext
    with open(file_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image_url", "image_url": {"url": f"data:image/{mime};base64,{b64}"}},
            ],
        }],
    )
    return resp.choices[0].message.content

## Prompt

Answers are graded by comparing them exactly with the expected answer. If the right answer is `3` and the agent answers `3 albums`, it counts as wrong.

That's why the prompt explains the answer format in detail: just the number, no units, lists separated by commas, etc.

In [ ]:
INSTRUCTIONS = """You are a general AI assistant solving a benchmark question.
Think step by step and use your tools (web search, visit webpages, Wikipedia, Python,
audio transcription, image analysis) to find the correct answer. Verify facts.

When you call final_answer, give ONLY the answer, nothing else:
- A number: digits only, no commas as thousands separators, no units ($, %) unless asked.
- A string: as few words as possible, no articles, no abbreviations.
- A list: a single comma-separated string, following the rules above for each element.
- Follow any format the question asks for exactly (e.g. alphabetical order, plural form).
- Never write 'FINAL ANSWER' or explanations.
- Never invent or simulate data, files or transcriptions. If a file path is given, use exactly that path.
- If a webpage is blocked, try another source instead of guessing.
- In each Thought, check whether the last Observation actually answers the question before calling final_answer."""

## The agent

The `AtlasAgent` class does three things:

1. Creates the `CodeAgent` with the model and the tools
2. Downloads the question's file when there is one (spreadsheet, audio, image). If the course API fails, it tries the GAIA dataset instead
3. Cleans up the final answer. For example, if it returns a list `['a', 'b']`, it turns it into `a, b`

In [ ]:
class AtlasAgent:
    def __init__(self, model_id: str = MODEL_ID, max_steps: int = MAX_STEPS):
        model = OpenAIServerModel(
            model_id=model_id,
            api_key=os.getenv("OPENAI_API_KEY"),
            client_kwargs={"timeout": 60, "max_retries": 2},
        )
        agent_kwargs = dict(
            model=model,
            tools=[
                DuckDuckGoSearchTool(),
                visit_webpage,
                WikipediaSearchTool(user_agent="Atlas-GAIA-agent (portfolio project)"),
                transcribe_audio,
                analyze_image,
            ],
            additional_authorized_imports=[
                "pandas", "numpy", "openpyxl", "json", "re", "math",
                "datetime", "collections", "itertools", "statistics",
            ],
            max_steps=max_steps,
        )
        try:
            self.agent = CodeAgent(**agent_kwargs, code_block_tags="markdown")
        except TypeError:  # older smolagents versions do not have this parameter
            self.agent = CodeAgent(**agent_kwargs)

    # ---------- attachments ----------
    def download_file(self, task_id: str, file_name: str) -> str | None:
        try:
            r = requests.get(f"{API_URL}/files/{task_id}", timeout=30)
            r.raise_for_status()
            path = os.path.join(tempfile.gettempdir(), file_name)
            with open(path, "wb") as f:
                f.write(r.content)
            return path
        except Exception as e:
            print(f"  API download failed ({e}); trying GAIA dataset...")
        try:
            return hf_hub_download(
                repo_id="gaia-benchmark/GAIA",
                repo_type="dataset",
                filename=f"2023/validation/{file_name}",
                token=os.getenv("HF_TOKEN"),
            )
        except Exception as e:
            print(f"  GAIA dataset download failed: {e}")
            return None

    # ---------- answer ----------
    @staticmethod
    def normalize(result) -> str:
        if isinstance(result, (list, tuple)):
            answer = ", ".join(str(x) for x in result)
        else:
            answer = str(result).strip()
        return re.sub(r"(?i)^final answer:\s*", "", answer).strip()

    def __call__(self, question: str, task_id: str | None = None, file_name: str | None = None) -> str:
        prompt = INSTRUCTIONS + "\n\nQuestion: " + question
        if file_name and task_id:
            path = self.download_file(task_id, file_name)
            if path:
                prompt += (
                    f"\n\nThe question has an attached file saved at: {path}\n"
                    "Use Python to read spreadsheets/code/text files, "
                    "transcribe_audio for audio and analyze_image for images."
                )
            else:
                prompt += "\n\nThe attached file could not be downloaded. Do not invent its content."
        try:
            return self.normalize(self.agent.run(prompt))
        except Exception as e:
            return f"AGENT ERROR: {e}"


atlas = AtlasAgent()
print("Agent ready")

You can see the instructions smolagents sends to the model behind the scenes. This is where it explains the think, act and observe loop:

In [ ]:
print(atlas.agent.prompt_templates["system_prompt"][:3000])

## Testing with one question

Before running everything, I test with a random question to check that it works.

In [ ]:
item = requests.get(f"{API_URL}/random-question", timeout=30).json()
print("Question:", item["question"])
print("File:", item.get("file_name") or "-")

answer = atlas(item["question"], item["task_id"], item.get("file_name"))
print("\nAtlas answer:", answer)

## Running all the questions

There are 20 questions. It takes about 10 to 30 minutes.

In [ ]:
questions = requests.get(f"{API_URL}/questions", timeout=30).json()
print(f"{len(questions)} questions")

results = []
for i, item in enumerate(questions, 1):
    print(f"\n[{i}/{len(questions)}] {item['question'][:100]}...")
    answer = atlas(item["question"], item["task_id"], item.get("file_name"))
    print("  ->", answer)
    results.append({
        "task_id": item["task_id"],
        "question": item["question"],
        "file_name": item.get("file_name") or "",
        "submitted_answer": answer,
    })

df = pd.DataFrame(results)
df.to_csv("atlas_answers.csv", index=False)
df[["question", "submitted_answer"]]

## Submitting the answers

To submit, the course API needs the Hugging Face username, the link to the code and the answers.

I left `SUBMIT = False` so it doesn't submit by accident.

In [ ]:
SUBMIT = False
HF_USERNAME = "your-hf-username"
AGENT_CODE_URL = "https://huggingface.co/spaces/your-hf-username/your-space/tree/main"

if SUBMIT:
    payload = {
        "username": HF_USERNAME,
        "agent_code": AGENT_CODE_URL,
        "answers": [{"task_id": r["task_id"], "submitted_answer": r["submitted_answer"]} for r in results],
    }
    resp = requests.post(f"{API_URL}/submit", json=payload, timeout=120)
    resp.raise_for_status()
    d = resp.json()
    print(f"Score: {d.get('score')}% ({d.get('correct_count')}/{d.get('total_attempted')} correct)")
    print(d.get("message", ""))
else:
    print("Submission disabled (SUBMIT = False).")

## What I learned

Some problems that came up while I was testing:

- **Wikipedia was blocking the agent.** I fixed it by writing my own tool to open pages, sending a `User-Agent` with the request.
- **The question files didn't download from the course API.** I added a fallback that downloads them from the GAIA dataset.
- **When a file was missing, the agent made up the answer.** I added a rule to the prompt saying it must not invent data.
- **Sometimes the answer came back as a Python list** (`['a', 'b']`), which counted as wrong. I wrote a function to fix the format.

## Next steps

- Answer questions about YouTube videos
- Try a version with more than one agent working together
- Replace DuckDuckGo with a more reliable search